# Analyse de Sentiment Darija — Exploration & Modélisation

Ce notebook part du dataset **déjà nettoyé et dédoublonné** (`dataset_pretraite.csv`,
produit par le pipeline de prétraitement). Il couvre :

1. Chargement du dataset nettoyé
2. Exploration & statistiques
3. Visualisations (histogramme, nuage de mots)
4. Entraînement et comparaison de 4 modèles de classification (TF-IDF + ML)

> Le nettoyage brut (suppression URLs/emojis, normalisation arabe, tokenisation,
> dédoublonnage) a déjà été effectué en amont et n'est pas répété ici.

## 0. Installation des dépendances

In [ ]:
!pip install -q pandas matplotlib wordcloud scikit-learn seaborn 2>/dev/null || pip install -q pandas matplotlib wordcloud scikit-learn seaborn --break-system-packages

## 1. Chargement du dataset nettoyé

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Chemin relatif : place dataset_pretraite.csv dans le même dossier que ce
# notebook (ou adapte le chemin, ex: 'data/dataset_pretraite.csv')
DATA_PATH = "dataset_pretraite.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df["text"] = df["text"].fillna("")

print(f"Dataset chargé : {len(df)} commentaires")
print(f"Colonnes : {df.columns.tolist()}")
df.head()

## 2. Analyse du bruit et des doublons

In [ ]:
total_comments     = len(df)
doublons           = df.duplicated(subset=["text"]).sum()
commentaires_vides = (df["text"].str.strip() == "").sum()

print("=== ANALYSE DU BRUIT ET DOUBLONS ===")
print(f"Total des commentaires               : {total_comments}")
print(f"Commentaires en double               : {doublons} ({(doublons/total_comments)*100:.2f}%)")
print(f"Commentaires vides ou inexploitables  : {commentaires_vides}")
print("-" * 50)

# Sécurité : si jamais il reste des doublons/vides, on les retire pour les stats suivantes
df_clean = df.drop_duplicates(subset=["text"])
df_clean = df_clean[df_clean["text"].str.strip() != ""].reset_index(drop=True)
print(f"Commentaires exploitables restants    : {len(df_clean)}")

## 3. Statistiques descriptives

In [ ]:
df_clean["longueur"] = df_clean["text"].apply(lambda x: len(str(x).split()))

tous_les_mots      = " ".join(df_clean["text"].tolist()).split()
vocabulaire_unique = set(tous_les_mots)

print("=== STATISTIQUES DESCRIPTIVES ===")
print(f"Commentaires exploitables          : {len(df_clean)}")
print(f"Longueur moyenne d'un commentaire  : {df_clean['longueur'].mean():.2f} mots")
print(f"Longueur maximale                  : {df_clean['longueur'].max()} mots")
print(f"Longueur minimale                  : {df_clean['longueur'].min()} mots")
print(f"Taille du vocabulaire unique        : {len(vocabulaire_unique)} mots")
print("-" * 50)

print("\nRépartition des sentiments :")
print(df_clean["sentiment"].value_counts())
print()
print((df_clean["sentiment"].value_counts(normalize=True) * 100).round(2))

## 4. Visualisations

In [ ]:
# ─── 4.1 Répartition des sentiments ──────────────────────────────────────
colors = {"positif": "#4CAF50", "négatif": "#F44336", "neutre": "#9E9E9E"}
counts = df_clean["sentiment"].value_counts()
bar_colors = [colors.get(l, "#2196F3") for l in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(counts.index, counts.values, color=bar_colors)
axes[0].set_title("Nombre de commentaires par sentiment")
axes[0].set_ylabel("Nombre de commentaires")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + len(df_clean) * 0.01, str(v), ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=bar_colors, startangle=90)
axes[1].set_title("Répartition des sentiments")

plt.tight_layout()
plt.savefig("repartition_sentiments.png", dpi=150)
plt.show()

In [ ]:
# ─── 4.2 Histogramme de la longueur des commentaires ─────────────────────
plt.figure(figsize=(10, 6))
plt.hist(df_clean["longueur"], bins=50, color="skyblue", edgecolor="black")
plt.title("Distribution de la longueur des commentaires (en nombre de mots)")
plt.xlabel("Nombre de mots")
plt.ylabel("Fréquence (Nombre de commentaires)")
plt.xlim(0, 100)
plt.grid(axis="y", alpha=0.75)
plt.tight_layout()
plt.savefig("histogramme_longueur.png", dpi=150)
plt.show()

In [ ]:
# ─── 4.3 Nuage de mots ────────────────────────────────────────────────────
# Les caractères arabes nécessitent une police qui les supporte (sinon ils
# s'affichent en rectangles vides). On cherche une police système adaptée,
# sinon on télécharge automatiquement Noto Sans Arabic (libre, Google Fonts).

import os
import urllib.request

def trouver_police_arabe():
    """Cherche une police supportant l'arabe, sinon la télécharge."""
    candidats_systeme = [
        "/usr/share/fonts/truetype/freefont/FreeSerif.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansArabic-Regular.ttf",
        "/usr/share/fonts/truetype/kacst/KacstOne.ttf",
        "/System/Library/Fonts/Supplemental/Tahoma.ttf",  # macOS
        "C:/Windows/Fonts/tahoma.ttf",                     # Windows
    ]
    for chemin in candidats_systeme:
        if os.path.exists(chemin):
            return chemin

    # Aucune police système trouvée -> téléchargement de Noto Sans Arabic
    chemin_local = "NotoSansArabic-Regular.ttf"
    if not os.path.exists(chemin_local):
        url = (
            "https://raw.githubusercontent.com/google/fonts/main/ofl/"
            "notosansarabic/NotoSansArabic%5Bwdth%2Cwght%5D.ttf"
        )
        try:
            urllib.request.urlretrieve(url, chemin_local)
        except Exception as e:
            print(f"Téléchargement de la police impossible : {e}")
            return None
    return chemin_local

font_path = trouver_police_arabe()
print(f"Police utilisée : {font_path}")

text_complet = " ".join(df_clean["text"].tolist())

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="viridis",
    max_words=200,
    font_path=font_path
).generate(text_complet)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Nuage de mots des commentaires", fontsize=16)
plt.tight_layout()
plt.savefig("nuage_de_mots.png", dpi=150)
plt.show()

## 5. Entraînement et évaluation des modèles

On entraîne 4 classifieurs sur une représentation **TF-IDF** :

| # | Modèle | Justification |
|---|--------|---------------|
| 1 | **Logistic Regression** | Baseline linéaire rapide, très interprétable |
| 2 | **SVM (LinearSVC)** | Souvent très performant sur texte haute dimension |
| 3 | **Naive Bayes** | Rapide, classique pour la classification de texte |
| 4 | **Random Forest** | Capture des interactions non linéaires |

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report)
from sklearn.pipeline import Pipeline
import numpy as np
import warnings
warnings.filterwarnings("ignore")

X = df_clean["text"]
y = df_clean["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {len(X_train)} | Test : {len(X_test)}")
print("Distribution train :")
print(y_train.value_counts())

In [ ]:
# Pipelines TF-IDF + Classifieur
tfidf_params = dict(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True,
    min_df=2
)

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf",  LogisticRegression(max_iter=1000, C=1.0, random_state=42))
    ]),
    "SVM (LinearSVC)": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf",  LinearSVC(C=1.0, max_iter=2000, random_state=42))
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf",  MultinomialNB(alpha=0.1))
    ]),
    "Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_params)),
        ("clf",  RandomForestClassifier(n_estimators=200, random_state=42))
        # Pas de n_jobs=-1 ici : éviterait un parallélisme imbriqué avec
        # cross_val_score(n_jobs=-1) plus bas, qui ralentit l'exécution.
    ]),
}

print("Pipelines définis :", list(models.keys()))

### Validation croisée 5-fold (robustesse)

On évalue les modèles avant l'entraînement final pour comparer de manière plus fiable.

In [ ]:
print("Validation croisée 5-fold (F1 macro) :\n")

for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X, y, cv=5, scoring="f1_macro", n_jobs=-1)
    print(f"{name:<22} : {scores.mean():.4f} ± {scores.std():.4f}  | scores={np.round(scores,4)}")

### Entraînement final et métriques détaillées

In [ ]:
results = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average="weighted")
    f1_m = f1_score(y_test, y_pred, average="macro")
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)

    results[name] = {
        "accuracy"   : acc,
        "f1_weighted": f1_w,
        "f1_macro"   : f1_m,
        "precision"  : prec,
        "recall"     : rec,
        "y_pred"     : y_pred
    }
    print(f"  ✅ {name:22} — Acc={acc*100:.1f}%  F1={f1_m*100:.1f}%  "
          f"Prec={prec*100:.1f}%  Rec={rec*100:.1f}%")

print("\n✅ Entraînement terminé pour tous les modèles")

### Matrices de confusion

In [ ]:
labels_order = sorted(y.unique())
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res["y_pred"], labels=labels_order)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_order, yticklabels=labels_order,
                ax=axes[i])
    axes[i].set_title(
        f"{name}\nAcc={res['accuracy']:.3f} | F1w={res['f1_weighted']:.3f}",
        fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Prédit")
    axes[i].set_ylabel("Réel")

plt.suptitle("Matrices de Confusion – Analyse de Sentiment Darija", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("matrices_confusion.png", dpi=150)
plt.show()

### Tableau comparatif des modèles

In [ ]:
summary = pd.DataFrame({
    name: {
        "Accuracy":    round(res["accuracy"], 4),
        "F1 Weighted": round(res["f1_weighted"], 4),
        "F1 Macro":    round(res["f1_macro"], 4),
    }
    for name, res in results.items()
}).T

print("=" * 55)
print("  TABLEAU COMPARATIF DES MODÈLES")
print("=" * 55)
print(summary.to_string())

best = summary["F1 Weighted"].idxmax()
print(f"\n✅ Meilleur modèle (F1 Weighted) : {best}")

summary_plot = summary[["Accuracy", "F1 Weighted", "F1 Macro"]].astype(float)
summary_plot.plot(kind="bar", figsize=(10, 5), edgecolor="black", colormap="viridis")
plt.title("Comparaison des modèles – Analyse de Sentiment Darija", fontsize=13)
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("comparaison_modeles.png", dpi=150)
plt.show()